In [ ]:
pip install selenium

In [ ]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options

options = Options()
options.add_argument("--headless=new")
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--disable-blink-features=AutomationControlled")
options.add_experimental_option("excludeSwitches", ["enable-automation"])
options.add_experimental_option("useAutomationExtension", False)

driver = webdriver.Chrome(options=options)
driver.get("https://www.google.com")
print("✅ Funcionando:", driver.title)
driver.quit()

In [ ]:
import time
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from bs4 import BeautifulSoup

options = Options()
# Testa SEM headless — janela visível tem menos chance de ser bloqueada
# options.add_argument("--headless=new")  # deixa comentado por enquanto
options.add_argument("--disable-blink-features=AutomationControlled")
options.add_argument("--window-size=1920,1080")
options.add_argument("--disable-extensions")
options.add_argument("--disable-popup-blocking")
options.add_argument("--profile-directory=Default")
options.add_argument("--disable-plugins-discovery")
options.add_experimental_option("excludeSwitches", ["enable-automation"])
options.add_experimental_option("useAutomationExtension", False)
options.add_argument(
    "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
    "AppleWebKit/537.36 (KHTML, like Gecko) "
    "Chrome/124.0.0.0 Safari/537.36"
)

driver = webdriver.Chrome(options=options)

# Oculta propriedades de automação
driver.execute_cdp_cmd("Page.addScriptToEvaluateOnNewDocument", {"source": """
    Object.defineProperty(navigator, 'webdriver', {get: () => undefined});
    Object.defineProperty(navigator, 'plugins', {get: () => [1, 2, 3]});
    Object.defineProperty(navigator, 'languages', {get: () => ['pt-BR', 'pt', 'en-US']});
    window.chrome = { runtime: {} };
"""})

# Acessa o Google antes para simular comportamento humano
driver.get("https://www.google.com.br")
time.sleep(2)

URL = (
    "https://www.tripadvisor.com.br/Attraction_Review-g680306-d2401618"
    "-Reviews-Praia_das_Laranjeiras-Balneario_Camboriu_State_of_Santa_Catarina.html"
)
driver.get(URL)
time.sleep(6)  # aguarda mais tempo

# Simula scroll para parecer humano
driver.execute_script("window.scrollTo(0, 300);")
time.sleep(1)
driver.execute_script("window.scrollTo(0, 600);")
time.sleep(1)

print("📄 Título da página:", driver.title)

soup = BeautifulSoup(driver.page_source, "html.parser")

h1s = soup.find_all("h1")
if h1s:
    print(f"\n✅ {len(h1s)} H1(s) encontrado(s):")
    for h1 in h1s:
        print(" -", h1.get_text(strip=True))
else:
    print("❌ Ainda bloqueado.")
    # Salva o HTML para inspecionar o que foi retornado
    with open("pagina_retornada.html", "w", encoding="utf-8") as f:
        f.write(driver.page_source)
    print("💾 HTML salvo em 'pagina_retornada.html' para inspeção.")

driver.quit()